<a href="https://colab.research.google.com/github/Yurnero2706/DataAlgo-UT/blob/main/DataAlgo2026_R10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# データ構造とアルゴリズム 遅延提出（第10週） (2026/06/24　ver.A)

---


★下記３要素を**必ず直接書き換えてから**提出すること．

* 学籍番号： 202318032
* 氏名： Nguyen Cong Nguyen
* Colabアカウント： ncnguyencva@gmail.com

**学生同士で教えた・教わった・グループワークをした場合，必ず下記の当該行を直接書き換えて記述すること．**
申告があった場合は，論述などで内容が似ていたり，プログラムが似ていても減点はしない．（コピペレベルの同一文などは処罰対象）
**教えた側は加点対象となる．**

**[教えた側]**
教えた相手：＜氏名＞　＜学籍番号＞
（何名いてもよい；教えた相手の内容が浅くなってないか確認すること）

**[教わった側]**
教わった相手：＜氏名＞　＜学籍番号＞
（何名いてもよい；必ず教えた側に自分の名前を書いてもらうこと）

**[グループワーク]**
一緒に行った相手：＜氏名＞　＜学籍番号＞
（何名いてもよいが，必ずお互いの名前を全員記すこと）

---
# 必須課題10A FPTASの概念説明

本授業内容に即して，FPTASアルゴリズムが有する性質について述べよ．
精度保証と計算量との関係については授業内での解説レベルで丁寧に説明すること．
その後，0-1ナップサック問題におけるFPTAS近似アルゴリズムについて，精度と計算量がどのような関係にあるか具体的に述べよ（導出は必要ない）




FPTAS stands for **Fully Polynomial-Time Approximation Scheme**. It has two nice properties at the same time:

1. **Accuracy on demand.**: For any $\varepsilon > 0$ that the user picks, the algorithm returns an answer within a factor of $(1 - \varepsilon)$ of the true optimum (for maximisation problems like knapsack).
2. **Fully polynomial time.**: The running time is polynomial in the input size $n$ **and** polynomial in $1/\varepsilon$ at the same time. As $\varepsilon$ becomes smaller, the cost only grows in a controlled way.

The important word is **"fully"**. Many approximation algorithms are polynomial in $n$ for a fixed $\varepsilon$, but their cost blows up exponentially in $1/\varepsilon$. So asking for a slightly tighter answer can make you wait much longer. An FPTAS avoids that as you can pick any $\varepsilon$, and pay a price that is polynomial in both $n$ and $1/\varepsilon$.

**0-1 knapsack FPTAS — the concrete numbers.**

For the 0-1 knapsack algorithm in §8.2 (the one that rounds each item's value and then solves a DP):

- **Accuracy:** the answer $V(X_a)$ is at least $(1 - \varepsilon)$ times the true optimum $V(X_{\text{opt}})$. So if $\varepsilon = 0.01$, you get an answer worth at least 99% of the best possible. If $\varepsilon = 0.001$, at least 99.9%. Whatever you want.
- **Time:** $O\!\left(\dfrac{n^3}{\varepsilon}\right)$.

Concretely, cutting $\varepsilon$ from 0.01 to 0.001 (10× tighter accuracy) costs only 10× more time — not $2^{10}$ times more. That is the FPTAS promise in action, which is **pay linearly for better accuracy, not exponentially**.

---
# 必須課題10B ミーリーマシンのファイルからの読み込み

ミーリーマシンを定義するテキストファイルを用意し，それを読み込んで実行するCプログラムを作成せよ．
入力文字・出力文字として考えるのは0-9の数字のみでよい．
状態は0以上の整数で表すこと．
受理状態については考慮しなくてよい．
用意するテキストファイルの形式については各自で考えてよいが，解説をつけること．
対象とするミーリーマシンは，授業内で用いていたものから利用してよい．
（他のミーリーマシンでもよいが，その場合は，そのミーリーマシンがどういう状態遷移を表現しているものか説明を用意すること）





**Mealy machine I am using.** I picked the **1-clock-delay** Mealy machine from the lecture (`FA-mealy.c`). It reads a stream of `0`/`1` digits and outputs each digit one step later. The first output is always `0` (because there is no previous input yet).

**File format.**

The file is a plain text file with three blocks of whitespace-separated integers.

1. **Header line** — three integers: `NUM_STATE NUM_INPUTLETTERS INIT_STATE`.
2. **Transition table** — `NUM_STATE` rows, each with `NUM_INPUTLETTERS` integers. Row $i$, column $j$ = the next state when the machine is in state $i$ and reads input letter $j$.
3. **Output table** — same shape as the transition table. Row $i$, column $j$ = the output letter emitted when the machine is in state $i$ and reads input letter $j$.

This format is simple to read with `fscanf("%d", ...)`. The order is fixed: header, then all transitions, then all outputs.

In [1]:
%%writefile mealy1clk.txt
3 2 0

1 2
1 2
1 2

0 0
0 0
1 1


Writing mealy1clk.txt


In [8]:
%%writefile FA-mealy-file.c
#include <stdio.h>
#include <stdlib.h>

#define MAX_STATE 256
#define MAX_LETTER 10

int NUM_STATE = 0;
int NUM_INPUTLETTERS = 0;
int INITSTATE = 0;

int ttable[MAX_STATE][MAX_LETTER];
int otable[MAX_STATE][MAX_LETTER];

int load_machine(const char *path) {
    FILE *fd = fopen(path, "r");
    if (!fd) { perror(path); return -1; }

    // Header (3 integers)
    if (fscanf(fd, "%d %d %d", &NUM_STATE, &NUM_INPUTLETTERS, &INITSTATE) != 3) {
        fprintf(stderr, "Failed to read header.\n");
        fclose(fd); return -2;
    }
    if (NUM_STATE <= 0 || NUM_STATE > MAX_STATE
        || NUM_INPUTLETTERS <= 0 || NUM_INPUTLETTERS > MAX_LETTER
        || INITSTATE < 0 || INITSTATE >= NUM_STATE) {
        fprintf(stderr, "Header values out of range.\n");
        fclose(fd); return -3;
    }

    // Transition table
    for (int s = 0; s < NUM_STATE; s++)
        for (int l = 0; l < NUM_INPUTLETTERS; l++)
            if (fscanf(fd, "%d", &ttable[s][l]) != 1) {
                fprintf(stderr, "Bad transition entry at (%d,%d).\n", s, l);
                fclose(fd); return -4;
            }

    // Output table
    for (int s = 0; s < NUM_STATE; s++)
        for (int l = 0; l < NUM_INPUTLETTERS; l++)
            if (fscanf(fd, "%d", &otable[s][l]) != 1) {
                fprintf(stderr, "Bad output entry at (%d,%d).\n", s, l);
                fclose(fd); return -5;
            }

    fclose(fd);
    return 0;
}

int mealymachine(int ss) {
    int c, t, cs, ns;
    cs = ss;
    printf("Input Output:\n 1st  input => ");
    while ((c = getc(stdin)) != EOF) {
        t = c - '0';
        if (t >= 0 && t < NUM_INPUTLETTERS) {
            ns = ttable[cs][t];
            printf("    %d      %d   \n next input => ", t, otable[cs][t]);
            cs = ns;
        }
    }
    return 0;
}

int main(int argc, char *argv[]) {
    if (argc != 2) {
        fprintf(stderr, "Usage: %s <machine_file.txt>\n", argv[0]);
        return -1;
    }
    if (load_machine(argv[1]) != 0) return -1;
    printf("Loaded machine: %d states, %d input letters, init = %d\n",
           NUM_STATE, NUM_INPUTLETTERS, INITSTATE);
    mealymachine(INITSTATE);
    return 0;
}


Overwriting FA-mealy-file.c


In [9]:
!gcc -Wall -o FA-mealy-file FA-mealy-file.c
!echo "00110110" | ./FA-mealy-file mealy1clk.txt

Loaded machine: 3 states, 2 input letters, init = 0
Input Output:
 1st  input =>     0      0   
 next input =>     0      0   
 next input =>     1      0   
 next input =>     1      1   
 next input =>     0      1   
 next input =>     1      0   
 next input =>     1      1   
 next input =>     0      1   
 next input => 

---
必須課題10C ミーリーマシンの実行の図解  

必須課題10Bで用意したミーリーマシンが，入力文字列に対して正しく動いていることを図解で説明せよ．  
図解で用いる用紙には，説明開始前に，予め当該ミーリーマシンを表す状態遷移図を描いておくこと（この部分は印刷でも手書きでもよい）．  
動画の最初で，当該ミーリーマシンの想定される機能について説明すること．  
その上で，入力文字列を用意し，その入力に伴う状態遷移を図の上で分かりやすく解説すること．  
YouTube Shortの形（２週目の課題参照）でテキストセル(2)にURLを示すこと．（手書きで紙に書くこと．肉声で解説しながら書くこと．本人確認も兼ねてるので，ペンタブや合成音声などは全て認めません）  
例 https://youtube.com/shorts/T2YnvHBectU  


https://youtube.com/shorts/eBJ4nJRtkVQ?feature=share

---
---
# 発展課題10X 新しいミーリーマシン（2クロック遅延）

2クロック遅延を実現するミーリーマシン表現を考え，FA-mealy.cのミーリーマシン定義部のみをそれに従って書き換えて実行し結果を確認せよ．またこのときの状態遷移図を示せ．

説明等はここに．

In [ ]:
プログラムコード．

---
# 課題提出法

筑波大学工学システム学類３年生向け．
FG24711 / FG34711．

必須課題は全て実施すること．
発展課題はしなくともよいが，A+取得には発展課題を（全課題提出を通して）1つ以上実施していることが必要条件である．

該当するmanabaに課題提出のエントリを設けるので，そこに本**Colab notebookを提出**すること．



# 出典

筑波大学工学システム学類  
データ構造とアルゴリズム  
担当：亀田能成  


---
2026/06/10 ver.A  
